# **Imports**

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import zipfile
import tensorflow as tf
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, Embedding, LSTM, Bidirectional, Dense, Dropout, Layer, Concatenate
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, classification_report
from sklearn.preprocessing import LabelEncoder
import re
import string
import seaborn as sns

In [ ]:
# Set random seeds for reproducibility
np.random.seed(42)
tf.random.set_seed(42)

# **Data Extraction Function**

In [ ]:
def extract_data(zip_path='data.zip'):
    """Extract the data from the zip file"""
    with zipfile.ZipFile(zip_path, 'r') as zip_ref:
        zip_ref.extractall('data')

    data_files = os.listdir('data')
    csv_files = [f for f in data_files if f.endswith('.csv')]

    if len(csv_files) > 0:
        return pd.read_csv(os.path.join('data', csv_files[0]))
    else:
        raise FileNotFoundError("No CSV file found in the extracted data.")

# **Text Cleaning**

In [ ]:
def clean_text(text):
    """Clean the text by removing special characters, extra spaces, etc."""
    if isinstance(text, str):
        text = text.lower()
        text = text.translate(str.maketrans('', '', string.punctuation))
        text = re.sub(r'\d+', '', text)
        text = re.sub(r'\s+', ' ', text).strip()
        return text
    return ""

# **Preprocessing**

In [ ]:
def preprocess_data(data, text_column='Poem', label_column='Genre', max_len=100, vocab_size=10000):
    """Preprocess the data for model training"""
    data = data.dropna(subset=[text_column, label_column])
    data[text_column] = data[text_column].apply(clean_text)
    label_encoder = LabelEncoder()
    encoded_labels = label_encoder.fit_transform(data[label_column])
    X_train, X_temp, y_train, y_temp = train_test_split(
        data[text_column], encoded_labels, test_size=0.3, random_state=42
    )

    X_val, X_test, y_val, y_test = train_test_split(
        X_temp, y_temp, test_size=0.5, random_state=42
    )
    tokenizer = Tokenizer(num_words=vocab_size, oov_token='<OOV>')
    tokenizer.fit_on_texts(X_train)
    X_train_seq = tokenizer.texts_to_sequences(X_train)
    X_val_seq = tokenizer.texts_to_sequences(X_val)
    X_test_seq = tokenizer.texts_to_sequences(X_test)
    X_train_pad = pad_sequences(X_train_seq, maxlen=max_len, padding='post')
    X_val_pad = pad_sequences(X_val_seq, maxlen=max_len, padding='post')
    X_test_pad = pad_sequences(X_test_seq, maxlen=max_len, padding='post')
    return (X_train_pad, y_train), (X_val_pad, y_val), (X_test_pad, y_test), tokenizer, label_encoder


# **Bahdanau Attention Layer**

In [ ]:
class BahdanauAttention(Layer):
    def __init__(self, units):
        super(BahdanauAttention, self).__init__()
        self.W1 = Dense(units)
        self.W2 = Dense(units)
        self.V = Dense(1)

    def call(self, query, values):
        query_with_time_axis = tf.expand_dims(query, 1)
        score = self.V(tf.nn.tanh(self.W1(query_with_time_axis) + self.W2(values)))
        attention_weights = tf.nn.softmax(score, axis=1)
        context_vector = attention_weights * values
        context_vector = tf.reduce_sum(context_vector, axis=1)
        return context_vector, attention_weights

# **Luong Attention Layer**

In [ ]:
class LuongAttention(Layer):
    def __init__(self, units, method='dot'):
        super(LuongAttention, self).__init__()
        self.method = method
        if method == 'general':
            self.W = Dense(units)
        elif method == 'concat':
            self.W = Dense(units)
            self.U = Dense(units)
            self.V = Dense(1)

    def call(self, query, values):
        if self.method == 'dot':
            score = tf.matmul(values, tf.expand_dims(query, -1))
        elif self.method == 'general':
            score = tf.matmul(values, tf.expand_dims(self.W(query), -1))
        elif self.method == 'concat':
            query_with_time_axis = tf.expand_dims(query, 1)
            score = self.V(tf.nn.tanh(self.W(query_with_time_axis) + self.U(values)))
        attention_weights = tf.nn.softmax(score, axis=1)
        context_vector = attention_weights * values
        context_vector = tf.reduce_sum(context_vector, axis=1)
        return context_vector, attention_weights

# **Bahdanau Model Builder**

In [ ]:
def build_bahdanau_model(vocab_size, embedding_dim, max_len, hidden_units=128, num_classes=1):
    inputs = Input(shape=(max_len,))
    embedded = Embedding(vocab_size, embedding_dim, input_length=max_len)(inputs)
    encoder_outputs = Bidirectional(LSTM(hidden_units, return_sequences=True))(embedded)
    query = tf.keras.layers.GlobalAveragePooling1D()(encoder_outputs)
    attention = BahdanauAttention(hidden_units)
    context_vector, attention_weights = attention(query, encoder_outputs)
    concat = Concatenate()([context_vector, query])
    x = Dense(hidden_units, activation='relu')(concat)
    x = Dropout(0.3)(x)

    if num_classes == 1:
        outputs = Dense(1, activation='sigmoid')(x)
    else:
        outputs = Dense(num_classes, activation='softmax')(x)


    model = Model(inputs=inputs, outputs=outputs)
    model_with_attention = Model(
        inputs=model.input,
        outputs=[model.output, attention_weights]
    )

    return model, model_with_attention

# **Luong Model Builder**

In [ ]:
def build_luong_model(vocab_size, embedding_dim, max_len, hidden_units=128, num_classes=1, method='dot'):
    inputs = Input(shape=(max_len,))
    embedded = Embedding(vocab_size, embedding_dim, input_length=max_len)(inputs)
    encoder_outputs = Bidirectional(LSTM(hidden_units, return_sequences=True))(embedded)
    query = tf.keras.layers.GlobalAveragePooling1D()(encoder_outputs)
    attention = LuongAttention(hidden_units, method=method)
    context_vector, attention_weights = attention(query, encoder_outputs)
    concat = Concatenate()([context_vector, query])
    x = Dense(hidden_units, activation='relu')(concat)
    x = Dropout(0.3)(x)

    if num_classes == 1:
        outputs = Dense(1, activation='sigmoid')(x)
    else:
        outputs = Dense(num_classes, activation='softmax')(x)
    model = Model(inputs=inputs, outputs=outputs)
    model_with_attention = Model(
        inputs=model.input,
        outputs=[model.output, attention_weights]
    )

    return model, model_with_attention

# **Training Function**

In [ ]:
def train_model(model, train_data, val_data, batch_size=32, epochs=10, model_name='model'):
    (X_train, y_train) = train_data
    (X_val, y_val) = val_data
    if len(np.unique(y_train)) == 2:
        loss = 'binary_crossentropy'
        metrics = ['accuracy']
    else:
        loss = 'sparse_categorical_crossentropy'
        metrics = ['accuracy']

    model.compile(optimizer='adam', loss=loss, metrics=metrics)

    early_stopping = EarlyStopping(monitor='val_loss', patience=3, restore_best_weights=True)
    model_checkpoint = ModelCheckpoint(f'{model_name}.h5', monitor='val_loss', save_best_only=True)

    history = model.fit(
        X_train, y_train,
        validation_data=(X_val, y_val),
        batch_size=batch_size,
        epochs=epochs,
        callbacks=[early_stopping, model_checkpoint]
    )

    return model, history

# **Evaluation Function**

In [ ]:
def evaluate_model(model, test_data, model_name='model'):
    (X_test, y_test) = test_data
    is_binary = len(np.unique(y_test)) == 2
    y_pred_proba = model.predict(X_test)

    if is_binary:
        y_pred = (y_pred_proba > 0.5).astype(int)
        accuracy = accuracy_score(y_test, y_pred)
        precision = precision_score(y_test, y_pred)
        recall = recall_score(y_test, y_pred)
        f1 = f1_score(y_test, y_pred)

        print(f"\nEvaluation Results for {model_name}:")
        print(f"Accuracy: {accuracy:.4f}")
        print(f"Precision: {precision:.4f}")
        print(f"Recall: {recall:.4f}")
        print(f"F1 Score: {f1:.4f}")

        print("\nClassification Report:")
        print(classification_report(y_test, y_pred))

        return {
            'accuracy': accuracy,
            'precision': precision,
            'recall': recall,
            'f1': f1
        }
    else:
        y_pred = np.argmax(y_pred_proba, axis=1)
        accuracy = accuracy_score(y_test, y_pred)
        precision = precision_score(y_test, y_pred, average='weighted')
        recall = recall_score(y_test, y_pred, average='weighted')
        f1 = f1_score(y_test, y_pred, average='weighted')

        print(f"\nEvaluation Results for {model_name}:")
        print(f"Accuracy: {accuracy:.4f}")
        print(f"Precision: {precision:.4f}")
        print(f"Recall: {recall:.4f}")
        print(f"F1 Score: {f1:.4f}")

        print("\nClassification Report:")
        print(classification_report(y_test, y_pred))

        return {
            'accuracy': accuracy,
            'precision': precision,
            'recall': recall,
            'f1': f1
        }

# **Plot Training History**

In [ ]:
def plot_history(history_bahdanau, history_luong):
    fig, axes = plt.subplots(1, 2, figsize=(15, 5))

    # Plot loss
    axes[0].plot(history_bahdanau.history['loss'], label='Bahdanau training')
    axes[0].plot(history_bahdanau.history['val_loss'], label='Bahdanau validation')
    axes[0].plot(history_luong.history['loss'], label='Luong training')
    axes[0].plot(history_luong.history['val_loss'], label='Luong validation')
    axes[0].set_title('Model Loss')
    axes[0].set_ylabel('Loss')
    axes[0].set_xlabel('Epoch')
    axes[0].legend()

    # Plot accuracy
    axes[1].plot(history_bahdanau.history['accuracy'], label='Bahdanau training')
    axes[1].plot(history_bahdanau.history['val_accuracy'], label='Bahdanau validation')
    axes[1].plot(history_luong.history['accuracy'], label='Luong training')
    axes[1].plot(history_luong.history['val_accuracy'], label='Luong validation')
    axes[1].set_title('Model Accuracy')
    axes[1].set_ylabel('Accuracy')
    axes[1].set_xlabel('Epoch')
    axes[1].legend()

    plt.tight_layout()
    plt.savefig('training_history.png')
    plt.close()

# **Visualize attention weights**

In [ ]:
def visualize_attention(model_with_attention, tokenizer, X_test, test_texts, indices, model_name='model'):
    for idx in indices:
        text = test_texts.iloc[idx]
        sequence = X_test[idx:idx+1]


        prediction, attention_weights = model_with_attention.predict(sequence)
        token_ids = sequence[0]
        tokens = []
        for id in token_ids:
            if id != 0:
                for word, idx in tokenizer.word_index.items():
                    if idx == id:
                        tokens.append(word)
                        break

        attn = attention_weights[0, :len(tokens), 0]
        plt.figure(figsize=(10, 4))
        plt.bar(range(len(tokens)), attn[:len(tokens)], align='center')
        plt.xticks(range(len(tokens)), tokens, rotation=45)
        plt.xlabel('Tokens')
        plt.ylabel('Attention Weight')
        if isinstance(prediction[0], np.ndarray):
            pred_text = f"Class: {np.argmax(prediction[0])}"
        else:
            pred_text = f"Score: {prediction[0][0]:.4f}"

        plt.title(f'{model_name} Attention Weights - {pred_text}')
        plt.tight_layout()
        plt.savefig(f'{model_name}_attention_sample_{idx}.png')
        plt.close()

# **Compare attention maps**

In [ ]:
def compare_attention_maps(bahdanau_model, luong_model, tokenizer, X_test, test_texts, indices):
    for idx in indices:
        text = test_texts.iloc[idx]
        sequence = X_test[idx:idx+1]
        bahdanau_pred, bahdanau_attn = bahdanau_model.predict(sequence)
        luong_pred, luong_attn = luong_model.predict(sequence)
        token_ids = sequence[0]
        tokens = []
        for id in token_ids:
            if id != 0:
                for word, idx in tokenizer.word_index.items():
                    if idx == id:
                        tokens.append(word)
                        break

        bahdanau_weights = bahdanau_attn[0, :len(tokens), 0]
        luong_weights = luong_attn[0, :len(tokens), 0]
        plt.figure(figsize=(12, 6))

        plt.subplot(2, 1, 1)
        plt.bar(range(len(tokens)), bahdanau_weights[:len(tokens)], align='center')
        plt.xticks(range(len(tokens)), tokens, rotation=45)
        if isinstance(bahdanau_pred[0], np.ndarray):
            bahdanau_pred_text = f"Class: {np.argmax(bahdanau_pred[0])}"
        else:
            bahdanau_pred_text = f"Score: {bahdanau_pred[0][0]:.4f}"

        plt.title(f'Bahdanau Attention - {bahdanau_pred_text}')

        plt.subplot(2, 1, 2)
        plt.bar(range(len(tokens)), luong_weights[:len(tokens)], align='center')
        plt.xticks(range(len(tokens)), tokens, rotation=45)
        if isinstance(luong_pred[0], np.ndarray):
            luong_pred_text = f"Class: {np.argmax(luong_pred[0])}"
        else:
            luong_pred_text = f"Score: {luong_pred[0][0]:.4f}"

        plt.title(f'Luong Attention - {luong_pred_text}')

        plt.tight_layout()
        plt.savefig(f'comparison_attention_sample_{idx}.png')
        plt.close()


# **Main execution**

In [ ]:
def main():
    # Extract data
    try:
        data = extract_data()
        print("Data loaded successfully!")
        print(f"Dataset shape: {data.shape}")
        print(f"Columns: {data.columns.tolist()}")

        # Check for missing values
        missing = data.isnull().sum()
        print("Missing values:")
        print(missing)

        # Set correct columns
        print("Using 'Poem' as text column and 'Genre' as label column")
        text_column = 'Poem'
        label_column = 'Genre'

        # Verify the columns exist
        if text_column not in data.columns:
            print(f"Text column '{text_column}' not found in dataset. Available columns: {data.columns.tolist()}")
            return

        if label_column not in data.columns:
            print(f"Label column '{label_column}' not found in dataset. Available columns: {data.columns.tolist()}")
            return

        # Print sample data
        print("\nSample data:")
        print(data[[text_column, label_column]].head(3))

        # Preprocess data
        print("\nPreprocessing data...")
        (X_train, y_train), (X_val, y_val), (X_test, y_test), tokenizer, label_encoder = preprocess_data(
            data, text_column=text_column, label_column=label_column
        )

        # Hyperparameters
        vocab_size = min(10000, len(tokenizer.word_index) + 1)
        embedding_dim = 128
        max_len = X_train.shape[1]
        hidden_units = 128
        batch_size = 32
        epochs = 10

        # Get number of classes
        num_classes = len(label_encoder.classes_)

        print(f"Vocabulary size: {vocab_size}")
        print(f"Maximum sequence length: {max_len}")
        print(f"Number of classes: {num_classes}")
        print(f"Classes: {label_encoder.classes_}")

        # Build models
        print("\nBuilding models...")
        bahdanau_model, bahdanau_model_with_attention = build_bahdanau_model(
            vocab_size, embedding_dim, max_len, hidden_units, num_classes
        )

        luong_model, luong_model_with_attention = build_luong_model(
            vocab_size, embedding_dim, max_len, hidden_units, num_classes
        )

        # Print model summaries
        print("\nBahdanau Model Summary:")
        bahdanau_model.summary()

        print("\nLuong Model Summary:")
        luong_model.summary()

        # Train models
        print("\nTraining Bahdanau Model...")
        bahdanau_model, bahdanau_history = train_model(
            bahdanau_model, (X_train, y_train), (X_val, y_val),
            batch_size=batch_size, epochs=epochs, model_name='bahdanau'
        )

        print("\nTraining Luong Model...")
        luong_model, luong_history = train_model(
            luong_model, (X_train, y_train), (X_val, y_val),
            batch_size=batch_size, epochs=epochs, model_name='luong'
        )

        # Evaluate models
        print("\nEvaluating Models...")
        bahdanau_metrics = evaluate_model(bahdanau_model, (X_test, y_test), model_name='Bahdanau')
        luong_metrics = evaluate_model(luong_model, (X_test, y_test), model_name='Luong')

        # Plot training history
        plot_history(bahdanau_history, luong_history)

        # Visualize attention
        print("\nVisualizing Attention Maps...")
        # Get indices for test samples
        test_indices = data.index[len(X_train)+len(X_val):]
        if len(test_indices) > 3:
            sample_indices = np.random.choice(len(X_test), size=3, replace=False)
        else:
            sample_indices = range(len(X_test))

        visualize_attention(
            bahdanau_model_with_attention, tokenizer,
            X_test, data.loc[test_indices, text_column],
            sample_indices, model_name='bahdanau'
        )

        visualize_attention(
            luong_model_with_attention, tokenizer,
            X_test, data.loc[test_indices, text_column],
            sample_indices, model_name='luong'
        )

        compare_attention_maps(
            bahdanau_model_with_attention, luong_model_with_attention,
            tokenizer, X_test, data.loc[test_indices, text_column],
            sample_indices
        )

        # Compare model performance
        print("\nModel Comparison:")
        comparison = pd.DataFrame({
            'Bahdanau': bahdanau_metrics,
            'Luong': luong_metrics
        })
        print(comparison)

        # Save comparison to CSV
        comparison.to_csv('model_comparison.csv')

        # Analysis and conclusion
        print("\nAttention Mechanism Analysis:")
        if bahdanau_metrics['accuracy'] > luong_metrics['accuracy']:
            print("The Bahdanau (Additive) Attention mechanism performed better in terms of accuracy.")
        elif luong_metrics['accuracy'] > bahdanau_metrics['accuracy']:
            print("The Luong (Multiplicative) Attention mechanism performed better in terms of accuracy.")
        else:
            print("Both attention mechanisms performed similarly in terms of accuracy.")

        print("\nAttention Weight Analysis:")
        print("The attention visualizations show which words each model focused on when making predictions.")
        print("This provides interpretability by highlighting the most important parts of the input sequence.")

    except Exception as e:
        import traceback
        print(f"An error occurred: {str(e)}")
        print(traceback.format_exc())

if __name__ == "__main__":
    main()

Data loaded successfully!
Dataset shape: (841, 2)
Columns: ['Genre', 'Poem']
Missing values:
Genre    0
Poem     4
dtype: int64
Using 'Poem' as text column and 'Genre' as label column

Sample data:
                                                Poem  Genre
0                                                NaN  Music
1                In the thick brushthey spend the...  Music
2     Storms are generous.                       ...  Music

Preprocessing data...


<ipython-input-16-60a42012e05c>:8: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  data[text_column] = data[text_column].apply(clean_text)
/usr/local/lib/python3.11/dist-packages/keras/src/layers/core/embedding.py:90: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


Vocabulary size: 7434
Maximum sequence length: 100
Number of classes: 4
Classes: ['Affection' 'Death' 'Environment' 'Music']

Building models...

Bahdanau Model Summary:


Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer         │ (None, 100)       │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ embedding           │ (None, 100, 128)  │    951,552 │ input_layer[0][0] │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ bidirectional       │ (None, 100, 256)  │    263,168 │ embedding[0][0]   │
│ (Bidirectional)     │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ global_average_poo… │ (None, 256)       │          0 │ bidirectional[0]… │
│ (GlobalAveragePool… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ bahdanau_attention  │ [(None, 256),     │     65,921 │ global_average_p… │
│ (BahdanauAttention) │ (None, 100, 1)]   │            │ bidirectional[0]… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ concatenate         │ (None, 512)       │          0 │ bahdanau_attenti… │
│ (Concatenate)       │                   │            │ global_average_p… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_3 (Dense)     │ (None, 128)       │     65,664 │ concatenate[0][0] │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout (Dropout)   │ (None, 128)       │          0 │ dense_3[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_4 (Dense)     │ (None, 4)         │        516 │ dropout[0][0]     │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

...

4/4 ━━━━━━━━━━━━━━━━━━━━ 2s 374ms/step

Evaluation Results for Luong:
Accuracy: 0.3095
Precision: 0.1834
Recall: 0.3095
F1 Score: 0.2118

Classification Report:
              precision    recall  f1-score   support

           0       0.00      0.00      0.00        22
           1       0.36      0.33      0.34        40
           2       0.29      0.87      0.43        30
           3       0.00      0.00      0.00        34

    accuracy                           0.31       126
   macro avg       0.16      0.30      0.19       126
weighted avg       0.18      0.31      0.21       126



/usr/local/lib/python3.11/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.11/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.11/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.11/dist-packages/sklearn/m


Visualizing Attention Maps...


1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 1s/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 127ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 119ms/step


1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 740ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 98ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 112ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 106ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 124ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 60ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 60ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 211ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 159ms/step

Model Comparison:
           Bahdanau     Luong
accuracy   0.325397  0.309524
precision  0.193686  0.183422
recall     0.325397  0.309524
f1         0.234413  0.211779

Attention Mechanism Analysis:
The Bahdanau (Additive) Attention mechanism performed better in terms of accuracy.

Attention Weight Analysis:
The attention visualizations show which words each model focused on when making predictions.
This provides interpretability by highlighting the most important parts of the input sequence.
